In [ ]:
# Per leggere i file
import gzip
import pickle

import numpy as np
from sktime.clustering.k_means import TimeSeriesKMeans
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

## Caricamento Dati
---

In [ ]:
with gzip.open("ts_normalized.pkl.gz", "rb") as f:
    ts_normalized = pickle.load(f)

with gzip.open("ts_transformed_paa.pkl.gz", "rb") as f:
    ts_transformed_paa = pickle.load(f)

## Whole Clustering
---
K-Means Euclideo è fattibile, K-Means con DTW è un mattone non runnabile I guess (da portatile)

In [ ]:
'''# Bisogna trasformare i dati in un array 3d come restituiva paa nel preprocessing
X = np.array(ts_normalized)
X_sktime = np.transpose(X, (0, 2, 1))

k = 5

# K Means Euclideo
kmeans_euclidean = TimeSeriesKMeans(
    n_clusters=k,
    metric="euclidean",
    random_state=222
)
labels_euclidean = kmeans_euclidean.fit_predict(X_sktime)
print("Clustering Euclideo completato.")

# K Means DTW
kmeans_dtw = TimeSeriesKMeans(
    n_clusters=k,
    metric="dtw",
    random_state=222
)
labels_dtw = kmeans_dtw.fit_predict(X_sktime)
print("Clustering DTW completato.")'''

## Approximate Clustering
---

In [ ]:
# Bisogna trasformare i dati in un array 3d come restituiva paa nel preprocessing
X = np.array(ts_transformed_paa)
X_sktime = np.transpose(X, (0, 2, 1))

k = 4

# K Means Euclideo

kmeans_euclidean = TimeSeriesKMeans(
    n_clusters=k,
    metric="euclidean",
    random_state=222
)

labels_euclidean = kmeans_euclidean.fit_predict(X_sktime)
print("Clustering Euclideo completato.")

# K Means DTW
kmeans_dtw = TimeSeriesKMeans(
    n_clusters=k,
    metric="dtw",
    random_state=222
)

labels_dtw = kmeans_dtw.fit_predict(X_sktime)
print("Clustering DTW completato.")

In [ ]:
# ==========================================
# 1. Preparazione e appiattimento dei dati
# ==========================================
X_flattened_list = []

for df in ts_transformed_paa:
    # L'attributo .values estrae l'array NumPy dal DataFrame.
    # Il metodo .flatten() converte la matrice (es. 20x13) in un vettore 1D.
    vettore_appiattito = df.values.flatten()
    X_flattened_list.append(vettore_appiattito)

# Conversione della lista in un array NumPy bidimensionale
X_flattened = np.array(X_flattened_list)

# Definizione del numero di cluster
k_clusters = 4

# ==========================================
# 2. Clustering Gerarchico (Agglomerativo)
# ==========================================
# L'algoritmo agglomerativo procede in modo deterministico unendo i cluster
# in base alle distanze matematiche esatte.
hierarchical_clusterer = AgglomerativeClustering(
    n_clusters=k_clusters,
    metric='euclidean',
    linkage='ward'
)

labels_ger = hierarchical_clusterer.fit_predict(X_flattened)

In [ ]:
import pacmap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

labels = labels_ger

# PCA
pca = PCA(n_components=2, random_state=222)
X_pca = pca.fit_transform(X_flattened)

# t-SNE
tsne = TSNE(n_components=2, random_state=222)
X_tsne = tsne.fit_transform(X_flattened)

# PacMAP
# L'inizializzazione con PCA è raccomandata per PacMAP per migliorare la stabilità
pm = pacmap.PaCMAP(n_components=2, random_state=222)
X_pacmap = pm.fit_transform(X_flattened, init="pca")

# ==========================================
# 2. Visualizzazione
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'Confronto Riduzione Dimensionalità - Approximate Clustering', fontsize=16)

# Grafico PCA
scatter_pca = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels,
                              cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5)
axes[0].set_title('PCA')
axes[0].set_xlabel('Componente Principale 1')
axes[0].set_ylabel('Componente Principale 2')

# Grafico t-SNE
scatter_tsne = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels,
                               cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5)
axes[1].set_title('t-SNE')
axes[1].set_xlabel('Dimensione 1')
axes[1].set_ylabel('Dimensione 2')

# Grafico PacMAP
scatter_pacmap = axes[2].scatter(X_pacmap[:, 0], X_pacmap[:, 1],
                                 c=labels, cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5)
axes[2].set_title('PacMAP')
axes[2].set_xlabel('Dimensione 1')
axes[2].set_ylabel('Dimensione 2')

# Aggiunta di una legenda per i cluster
# Utilizza le informazioni dell'ultimo scatter plot per generare la legenda
legend = fig.legend(*scatter_pacmap.legend_elements(),
                    title="Cluster", loc='center right')
fig.subplots_adjust(right=0.92)  # Spazio per la legenda

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
# ==========================================
# 1. Validazione per K-Means (Euclideo)
# ==========================================
k_range = range(2, 11)
inertia_values = []
silhouette_scores = []

for k_val in k_range:
    kmeans_eval = TimeSeriesKMeans(
        n_clusters=k_val, 
        metric="euclidean", 
        random_state=222
    )
    # L'algoritmo di sktime riceve l'array 3D
    labels_eval = kmeans_eval.fit_predict(X_sktime)
    
    inertia_values.append(kmeans_eval.inertia_)
    
    # La metrica Silhouette di scikit-learn richiede array 2D
    score = silhouette_score(X_flattened, labels_eval, metric='euclidean')
    silhouette_scores.append(score)

# Visualizzazione Metodo del Gomito e Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertia_values, marker='o', color='blue')
ax1.set_title('Metodo del Gomito (Elbow Method)')
ax1.set_xlabel('Numero di Cluster (k)')
ax1.set_ylabel('Inerzia')
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(k_range, silhouette_scores, marker='o', color='orange')
ax2.set_title('Indice di Silhouette')
ax2.set_xlabel('Numero di Cluster (k)')
ax2.set_ylabel('Silhouette Score Medio')
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# ==========================================
# 2. Validazione per Clustering Gerarchico
# ==========================================
# Il linkage 'ward' calcola le distanze in base alla varianza ed è coerente con l'implementazione
Z = linkage(X_flattened, method='ward')

plt.figure(figsize=(12, 7))
plt.title("Dendrogramma per la Scelta di K")
# Troncamento visivo per evitare un albero illeggibile in presenza di molti campioni
dendrogram(Z, truncate_mode='lastp', p=40)
plt.xlabel("Dimensione dei cluster / Indici dei campioni")
plt.ylabel("Distanza di Ward")
plt.show()